# Phase 4b: Training & Evaluation

### Fixes vs previous version:
1. **No data leakage**: train with `gnn_features_train.pkl`, eval with `gnn_features_test.pkl`
2. **Actual budget_ratio/step_ratio** in pretrain (not hardcoded)
3. **Reward shaping** implemented (MINERVA-style potential-based)
4. **GreedyGNN baseline** evaluated for comparison
5. **Both approaches** trained: Traversal RL + Path Selection RL

### Training strategy:
- **Stage 1**: Imitation Learning (Behavior Cloning from GNN-RAG paths)
- **Stage 2**: REINFORCE with baseline + reward shaping

In [49]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/ThS/Research/colab/graph_rag')
from common import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import pickle
import json
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
# === Import all classes from grl_rag.py (shared module) ===
import sys, os, shutil, importlib

GRL_RAG_PATH = '/content/drive/MyDrive/ThS/Research/colab/graph_rag'
GRL_RAG_FILE = os.path.join(GRL_RAG_PATH, 'grl_rag.py')

# 1. Verify file exists and is readable
assert os.path.exists(GRL_RAG_FILE), f'File not found: {GRL_RAG_FILE}'
size = os.path.getsize(GRL_RAG_FILE)
print(f'grl_rag.py found: {size} bytes')

# 2. Delete __pycache__ to clear any cached failed imports
pycache = os.path.join(GRL_RAG_PATH, '__pycache__')
if os.path.exists(pycache):
    shutil.rmtree(pycache)
    print('Deleted __pycache__')

# 3. Clean ALL grl_rag references from sys.modules
to_del = [k for k in sys.modules if 'grl_rag' in k]
for k in to_del:
    del sys.modules[k]
    print(f'Removed {k} from sys.modules')

# 4. Add path
if GRL_RAG_PATH not in sys.path:
    sys.path.insert(0, GRL_RAG_PATH)
    print(f'Added to sys.path: {GRL_RAG_PATH}')

# 5. Try import with detailed error
try:
    import grl_rag
    print(f'Import OK: {grl_rag.__file__}')
except Exception as e:
    print(f'Import FAILED: {type(e).__name__}: {e}')
    # Try to compile the file to check for syntax errors
    print('\nChecking for syntax errors...')
    with open(GRL_RAG_FILE) as f:
        code = f.read()
    try:
        compile(code, GRL_RAG_FILE, 'exec')
        print('No syntax errors - problem is a runtime import error inside the file')
        print('First 5 lines:')
        for i, line in enumerate(code.split('\n')[:5]):
            print(f'  {i+1}: {line}')
    except SyntaxError as se:
        print(f'SYNTAX ERROR at line {se.lineno}: {se.msg}')
    raise

from grl_rag import *

EXPECTED = [
    'GNNHeuristic', 'Embedder', 'GreedyGNNRetriever',
    'TraversalPolicy', 'TraversalRetriever',
    'PathSelectionPolicy', 'PathSelectionRetriever',
    'BudgetKGEnvironment', 'compute_reward',
    'path_to_string', 'extract_candidate_paths',
]
missing = [name for name in EXPECTED if name not in dir()]
if missing:
    raise ImportError(f'Missing: {missing}')
print(f'OK: All {len(EXPECTED)} classes/functions imported')



grl_rag.py found: 22419 bytes
Deleted __pycache__
Removed grl_rag from sys.modules
Import OK: /content/drive/MyDrive/ThS/Research/research/COLAB/RUNNING/utils/grl_rag.py
OK: All 11 classes/functions imported


## 1. Load Data (NO LEAKAGE)

**CRITICAL**: Use TRAIN GNN features for training, TEST for evaluation only.

In [51]:
# FIX: Separate train/test GNN features
gnn_train = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_train.pkl'))
gnn_test = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl'))

print(f'Train GNN features: {len(gnn_train.features)} questions')
print(f'Test GNN features: {len(gnn_test.features)} questions')

if not gnn_train.features:
    print('\nWARNING: No train features! Run P2 cell "6b" to generate train.info first.')

# Embedder
EMB_CACHE = os.path.join(CHECKPOINT_DIR, 'embeddings_cache.pkl')
embedder = Embedder(cache_path=EMB_CACHE)
EMB_DIM = embedder.emb_dim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Loaded GNN features for 2848 questions
Loaded GNN features for 1639 questions
Train GNN features: 2848 questions
Test GNN features: 1639 questions


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


## 2. Evaluate GreedyGNN Baseline (no training needed)

In [52]:
def evaluate_retriever(retriever, gnn, max_samples=None):
    """Evaluate any retriever. Check entity overlap with gold answers."""
    results = []
    qids = list(gnn.features.keys())
    if max_samples:
        qids = qids[:max_samples]

    for qid in tqdm(qids, desc='Evaluating'):
        qdata = gnn.get(qid)
        gold = set(gnn.get_answers(qid))

        r = retriever.retrieve(qid) if not hasattr(retriever, 'policy') else retriever.retrieve(qid, greedy=True)

        collected = set()
        for h, rel, t in r['triples']:
            collected.add(str(h))
            collected.add(str(t))

        overlap = collected & gold
        hits = 1.0 if overlap else 0.0
        recall = len(overlap) / max(len(gold), 1)
        precision = len(overlap) / max(len(collected), 1)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        results.append({'qid': qid, 'f1': f1, 'hits': hits, 'recall': recall,
                        'cost': r['cost'], 'n_triples': len(r['triples'])})

    summary = {
        'n': len(results),
        'f1': np.mean([r['f1'] for r in results]),
        'hits': np.mean([r['hits'] for r in results]),
        'recall': np.mean([r['recall'] for r in results]),
        'cost': np.mean([r['cost'] for r in results]),
    }
    print(f'  N={summary["n"]} | F1={summary["f1"]:.4f} | Hits={summary["hits"]:.4f} | '
          f'Recall={summary["recall"]:.4f} | Cost={summary["cost"]:.1f}')
    return summary, results


# === Greedy GNN Baseline (no training) ===
print('=== GreedyGNN Baseline (test set) ===')
greedy_retriever = GreedyGNNRetriever(gnn_test, max_steps=5, budget=50)
greedy_summary, greedy_results = evaluate_retriever(greedy_retriever, gnn_test, max_samples=500)

with open(os.path.join(RESULTS_DIR, 'greedy_gnn_webqsp.json'), 'w') as f:
    json.dump({'summary': greedy_summary}, f, indent=2)
print('Saved greedy baseline results.')

=== GreedyGNN Baseline (test set) ===


Evaluating: 100%|██████████| 500/500 [00:12<00:00, 41.06it/s] 

  N=500 | F1=0.0776 | Hits=0.6580 | Recall=0.5648 | Cost=49.9
Saved greedy baseline results.


## 3. Stage 1: Imitation Learning (FIX: actual budget_ratio)

In [53]:
def extract_expert_trajectories(gnn, max_samples=None):
    """Extract expert trajectories from TRAIN GNN features."""
    trajectories = []
    qids = list(gnn.features.keys())
    if max_samples:
        qids = qids[:max_samples]

    for qid in tqdm(qids, desc='Extracting expert trajectories'):
        qdata = gnn.get(qid)
        G = gnn.get_local_graph(qid)
        if G.number_of_edges() == 0:
            continue

        seeds = gnn.get_seeds(qid)
        candidates = qdata.get('top_candidates', [])[:10]
        if not seeds or not candidates:
            continue

        for seed in seeds:
            if seed not in G:
                continue
            for cand in candidates:
                if cand not in G or cand == seed:
                    continue
                try:
                    path = nx.shortest_path(G, seed, cand)
                except nx.NetworkXNoPath:
                    continue
                if len(path) < 2:
                    continue

                actions = []
                for i in range(len(path) - 1):
                    cur, nxt = path[i], path[i + 1]
                    neighbors = list(G.neighbors(cur))
                    if nxt in neighbors:
                        rel = G[cur][nxt].get('relation', 'related_to')
                        actions.append({
                            'current': cur, 'next': nxt, 'relation': rel,
                            'neighbors': neighbors,
                            'step_in_path': i,
                            'total_steps': len(path) - 1,
                        })

                if actions:
                    trajectories.append({'qid': qid, 'question': qdata['question'],
                                        'actions': actions})

    print(f'Extracted {len(trajectories)} trajectories')
    return trajectories


# FIX: Use TRAIN data
expert_trajs = extract_expert_trajectories(gnn_train)

Extracting expert trajectories: 100%|██████████| 2848/2848 [00:20<00:00, 136.20it/s]

Extracted 11225 trajectories


In [54]:
def pretrain_traversal(policy, embedder, expert_trajs, gnn,
                        num_epochs=10, lr=1e-3, budget=50,
                        max_steps=10, device='cpu'):
    """IL pre-training with ACTUAL budget_ratio and relation encoding."""
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    policy.train()

    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total = 0
        np.random.shuffle(expert_trajs)

        for traj in expert_trajs:
            qid = traj['qid']
            query_emb = embedder.encode(traj['question']).to(device)
            visited = [traj['actions'][0]['current']]

            for step in traj['actions']:
                cur = step['current']
                target = step['next']
                neighbors = step['neighbors']
                if len(neighbors) <= 1 or target not in neighbors:
                    continue

                # Build action list: neighbors + STOP
                act_ent = []
                act_rel = []
                gnn_scores = []
                G = gnn.get_local_graph(qid)

                for nb in neighbors:
                    act_ent.append(embedder.encode(nb))
                    rel = G[cur][nb].get('relation', 'related_to') if G.has_edge(cur, nb) else 'related_to'
                    act_rel.append(embedder.encode(rel))
                    gnn_scores.append(gnn.get_node_score(qid, nb))

                # STOP action
                act_ent.append(policy.stop_entity_emb.detach().cpu())
                act_rel.append(policy.stop_relation_emb.detach().cpu())
                gnn_scores.append(0.0)

                act_ent_t = torch.stack(act_ent).unsqueeze(0).to(device)
                act_rel_t = torch.stack(act_rel).unsqueeze(0).to(device)
                gnn_t = torch.tensor(gnn_scores).unsqueeze(0).unsqueeze(-1).float().to(device)

                node_emb = embedder.encode(cur).unsqueeze(0).to(device)
                hist = embedder.encode_batch(visited).unsqueeze(0).to(device)

                # FIX: actual budget/step ratios
                b_ratio = torch.tensor([[max(0, (budget - len(visited)) / budget)]]).to(device)
                s_ratio = torch.tensor([[step['step_in_path'] / max_steps]]).to(device)

                probs = policy(query_emb.unsqueeze(0), node_emb, hist,
                              b_ratio, s_ratio, act_ent_t, act_rel_t, gnn_t)

                expert_idx = neighbors.index(target)
                loss = F.nll_loss(torch.log(probs + 1e-8),
                                 torch.tensor([expert_idx], device=device))

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()

                total_loss += loss.item()
                correct += (probs.argmax().item() == expert_idx)
                total += 1
                visited.append(target)

        avg_loss = total_loss / max(total, 1)
        acc = correct / max(total, 1)
        print(f'Epoch {epoch+1}/{num_epochs}: Loss={avg_loss:.4f}, Acc={acc:.4f}')

        torch.save({'epoch': epoch+1, 'model_state': policy.state_dict(),
                    'loss': avg_loss, 'acc': acc},
                   os.path.join(CHECKPOINT_DIR, f'traversal_pretrain_e{epoch+1}.pt'))

    return policy


print('pretrain_traversal defined (with actual budget_ratio + relation encoding).')

pretrain_traversal defined (with actual budget_ratio + relation encoding).


In [55]:
# === Run pre-training ===
policy_traversal = TraversalPolicy(emb_dim=EMB_DIM).to(device)
policy_traversal = pretrain_traversal(
    policy_traversal, embedder, expert_trajs, gnn_train,
    num_epochs=10, lr=1e-3, device=device
)

print('Pre-training (uncomment when ready)')

Epoch 1/10: Loss=3.0500, Acc=0.3379
Epoch 2/10: Loss=3.0072, Acc=0.3546
Epoch 3/10: Loss=3.0199, Acc=0.3611
Epoch 4/10: Loss=2.8459, Acc=0.3614
Epoch 5/10: Loss=2.5062, Acc=0.3637
Epoch 6/10: Loss=2.4174, Acc=0.3720
Epoch 7/10: Loss=2.3450, Acc=0.3754
Epoch 8/10: Loss=2.2982, Acc=0.3799
Epoch 9/10: Loss=2.2569, Acc=0.3795
Epoch 10/10: Loss=2.2259, Acc=0.3855
Pre-training (uncomment when ready)


## 4. Stage 2: RL Fine-tuning (with reward shaping)

In [ ]:
class RLTrainer:
    """REINFORCE with baseline + reward shaping (MINERVA-style)."""

    def __init__(self, retriever, gnn, embedder,
                 lr=1e-4, gamma=0.99, use_shaping=True,
                 reward_weights=(0.6, 0.2, 0.2), device='cpu'):
        self.retriever = retriever
        self.gnn = gnn
        self.embedder = embedder
        self.gamma = gamma
        self.use_shaping = use_shaping
        self.alpha, self.beta, self.gamma_w = reward_weights
        self.device = device

        self.optimizer = torch.optim.Adam(retriever.policy.parameters(), lr=lr)
        self.baseline = 0.0
        self.baseline_decay = 0.99
        self.log = {'rewards': [], 'losses': [], 'costs': []}

    def _reward_shaping_potential(self, qid, entities):
        """Potential function phi(s) = max GNN score among entities.
        Used for MINERVA-style reward shaping.
        """
        if not entities:
            return 0.0
        scores = [self.gnn.get_node_score(qid, e) for e in entities]
        return max(scores) if scores else 0.0

    def train_episode(self, qid):
        self.retriever.policy.train()
        qdata = self.gnn.get(qid)
        if qdata is None:
            return 0.0, 0.0, 0

        result = self.retriever.retrieve(qid, greedy=False)
        if not result['log_probs']:
            return 0.0, 0.0, result['cost']

        gold = qdata.get('answers', [])
        terminal_reward = compute_reward(
            result['triples'], gold,
            result['cost'], self.retriever.budget
            if hasattr(self.retriever, 'budget') else self.retriever.max_paths,
            alpha=self.alpha, beta=self.beta, gamma=self.gamma_w
        )

        # Reward shaping (MINERVA-style)
        reward = terminal_reward
        if self.use_shaping and hasattr(self.retriever, 'env'):
            # Add potential-based shaping: gamma*phi(s') - phi(s)
            env = self.retriever.env
            path = env.path
            shaping_bonus = 0.0
            for i in range(1, len(path)):
                phi_prev = self._reward_shaping_potential(qid, path[:i])
                phi_curr = self._reward_shaping_potential(qid, path[:i+1])
                shaping_bonus += self.gamma * phi_curr - phi_prev
            reward += 0.1 * shaping_bonus  # Scale down shaping

        # REINFORCE
        advantage = reward - self.baseline
        self.baseline = self.baseline_decay * self.baseline + (1 - self.baseline_decay) * reward

        log_probs = torch.stack(result['log_probs'])
        loss = -(log_probs.sum() * advantage)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.retriever.policy.parameters(), 1.0)
        self.optimizer.step()

        self.log['rewards'].append(reward)
        self.log['losses'].append(loss.item())
        self.log['costs'].append(result['cost'])

        return loss.item(), reward, result['cost']

    def train(self, num_epochs=5, log_every=100, ckpt_prefix='rl'):
        qids = list(self.gnn.features.keys())

        for epoch in range(num_epochs):
            np.random.shuffle(qids)
            epoch_r, epoch_l = [], []

            for i, qid in enumerate(tqdm(qids, desc=f'Epoch {epoch+1}')):
                loss, reward, cost = self.train_episode(qid)
                epoch_r.append(reward)
                epoch_l.append(loss)

                if (i + 1) % log_every == 0:
                    print(f'  Step {i+1}: R={np.mean(epoch_r[-log_every:]):.4f}')

            print(f'Epoch {epoch+1}: R={np.mean(epoch_r):.4f}')
            torch.save({
                'epoch': epoch+1,
                'model_state': self.retriever.policy.state_dict(),
                'baseline': self.baseline,
            }, os.path.join(CHECKPOINT_DIR, f'{ckpt_prefix}_epoch{epoch+1}.pt'))

        return self.log


print('RLTrainer defined (with ckpt_prefix to avoid overwrite).')

In [58]:
# === Hotfix: patch device mismatch ===
def _fixed_encode_actions(self, actions):
    entity_embs, relation_embs, scores = [], [], []
    for rel, entity, rel_text, gnn_score in actions:
        if entity is None:
            entity_embs.append(self.policy.stop_entity_emb.detach().cpu())
            relation_embs.append(self.policy.stop_relation_emb.detach().cpu())
        else:
            entity_embs.append(self.embedder.encode(entity))
            relation_embs.append(self.embedder.encode(rel_text))
        scores.append(gnn_score)
    return (torch.stack([e.cpu() for e in entity_embs]).unsqueeze(0).to(self.device),
            torch.stack([e.cpu() for e in relation_embs]).unsqueeze(0).to(self.device),
            torch.tensor(scores).unsqueeze(0).unsqueeze(-1).float().to(self.device))

TraversalRetriever._encode_actions = _fixed_encode_actions
print('Patched _encode_actions')

Patched _encode_actions


In [ ]:
# === Train Traversal RL ===
policy_traversal = TraversalPolicy(emb_dim=EMB_DIM).to(device)
# Load pretrain checkpoint first:
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'traversal_pretrain_e10.pt'))
policy_traversal.load_state_dict(ckpt['model_state'])

traversal_retriever = TraversalRetriever(
    gnn_train, policy_traversal, embedder,
    max_steps=10, budget=50, device=device
)
trainer = RLTrainer(traversal_retriever, gnn_train, embedder,
                    lr=1e-4, use_shaping=True, device=device)
trav_log = trainer.train(num_epochs=5, ckpt_prefix='traversal_rl')

print('Traversal RL training done. Checkpoints: traversal_rl_epoch{1-5}.pt')

In [ ]:
# === Train Path Selection RL ===
policy_selection = PathSelectionPolicy(emb_dim=EMB_DIM).to(device)
selection_retriever = PathSelectionRetriever(
    gnn_train, policy_selection, embedder,
    max_paths=10, device=device
)
trainer_sel = RLTrainer(selection_retriever, gnn_train, embedder,
                        lr=1e-4, use_shaping=False, device=device)
sel_log = trainer_sel.train(num_epochs=5, ckpt_prefix='selection_rl')

print('Selection RL training done. Checkpoints: selection_rl_epoch{1-5}.pt')

## 5. Final Evaluation (on TEST set)

In [61]:
# === Evaluate all methods on TEST set ===
# IMPORTANT: Use gnn_test (not gnn_train)

print('\n=== 1. GreedyGNN Baseline ===')
greedy = GreedyGNNRetriever(gnn_test, max_steps=5, budget=50)
greedy_s, _ = evaluate_retriever(greedy, gnn_test)

print('\n=== 2. Traversal RL ===')
trav = TraversalRetriever(gnn_test, policy_traversal, embedder,
                          max_steps=10, budget=50, device=device)
trav_s, _ = evaluate_retriever(trav, gnn_test)

print('\n=== 3. Path Selection RL ===')
sel = PathSelectionRetriever(gnn_test, policy_selection, embedder,
                             max_paths=10, device=device)
sel_s, _ = evaluate_retriever(sel, gnn_test)

print('\n=== Comparison ===')
import pandas as pd
df = pd.DataFrame([greedy_s, trav_s, sel_s],
                   index=['GreedyGNN', 'Traversal RL', 'Selection RL'])
print(df[['f1', 'hits', 'recall', 'cost']].to_markdown())

# Save results for P5
for name, summary, filename in [
    ('GreedyGNN', greedy_s, 'greedy_gnn_webqsp.json'),
    ('Traversal RL', trav_s, 'grl_traversal_webqsp.json'),
    ('Selection RL', sel_s, 'grl_selection_webqsp.json'),
]:
    with open(os.path.join(RESULTS_DIR, filename), 'w') as f:
        json.dump({'summary': summary}, f, indent=2)
    print(f'Saved: {filename}')



## 6. Checkpoint Management

In [62]:
def load_checkpoint(policy, path):
    ckpt = torch.load(path, map_location='cpu')
    policy.load_state_dict(ckpt['model_state'])
    print(f'Loaded: {path} (epoch {ckpt.get("epoch", "?")})')
    return ckpt.get('baseline', 0.0)


def find_latest(prefix, ckpt_dir=None):
    import glob
    d = ckpt_dir or CHECKPOINT_DIR
    files = glob.glob(os.path.join(d, f'{prefix}*.pt'))
    return max(files, key=os.path.getmtime) if files else None


print('Checkpoint utils defined.')

Checkpoint utils defined.


## Summary

### Fixes applied:
| Issue | Fix |
|-------|-----|
| Data leakage | `gnn_train.pkl` for training, `gnn_test.pkl` for eval |
| Hardcoded budget_ratio=0.8 | Actual `(budget - visited) / budget` |
| No reward shaping | MINERVA-style potential-based shaping |
| No baseline comparison | GreedyGNN evaluated first |
| Relation ignored in actions | Relation embedding added |
| Budget per seed | Shared budget across seeds |

### Results to compare in P5:
1. GreedyGNN (no RL, no training)
2. GRL-RAG Traversal (IL + RL on graph)
3. GRL-RAG Selection (RL selects paths)
4. GNN-RAG (published results)

In [63]:
# === Error Analysis: GNN Failure Cases ===
import json
from collections import Counter

# Load test features (da co trong memory tu cell truoc)
# Neu chua co: gnn_test = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl'))

qids = list(gnn_test.features.keys())

# Phan loai failures
failures = {'f1_zero': [], 'hit_zero_has_cand': [], 'no_candidates': []}
successes = []

for qid in qids:
    feat = gnn_test.get(qid)
    f1 = feat['gnn_f1']
    hit = feat['gnn_hit']
    cands = feat['top_candidates']
    answers = feat['answers']

    if len(cands) == 0:
        failures['no_candidates'].append(qid)
    elif f1 == 0:
        failures['f1_zero'].append(qid)
    elif hit == 0:
        failures['hit_zero_has_cand'].append(qid)
    else:
        successes.append(qid)

print('=== Failure Distribution ===')
print(f'Success (F1>0 & Hit=1): {len(successes)}')
print(f'F1=0 (total miss):      {len(failures["f1_zero"])}')
print(f'Hit=0 (wrong top-1):    {len(failures["hit_zero_has_cand"])}')
print(f'No candidates:           {len(failures["no_candidates"])}')

# === Chi tiet tung loai failure ===
print('\n' + '='*80)
print('=== TYPE 1: F1=0 (GNN khong tim duoc answer nao) ===')
print('='*80)
for qid in failures['f1_zero'][:10]:
    feat = gnn_test.get(qid)
    G = gnn_test.get_local_graph(qid)
    seeds = gnn_test.get_seeds(qid)
    answers = gnn_test.get_answers(qid)
    cands = feat['top_candidates'][:5]

    # Check: answer co trong graph khong?
    answers_in_graph = [a for a in answers if a in G]
    # Check: seed co trong graph khong?
    seeds_in_graph = [s for s in seeds if s in G]

    print(f'\n[{qid}] {feat["question"]}')
    print(f'  Seeds: {seeds} (in graph: {len(seeds_in_graph)}/{len(seeds)})')
    print(f'  Gold answers: {answers}')
    print(f'  Top-5 GNN candidates: {cands}')
    print(f'  Answers in graph: {len(answers_in_graph)}/{len(answers)}')
    if not answers_in_graph:
        print(f'  >> REASON: Gold answers NOT in subgraph!')
    else:
        print(f'  >> REASON: Answers in graph but GNN ranked them too low')

print('\n' + '='*80)
print('=== TYPE 2: Hit=0 but has candidates (top-1 sai, nhung co answer trong candidates) ===')
print('='*80)
for qid in failures['hit_zero_has_cand'][:10]:
    feat = gnn_test.get(qid)
    answers = set(gnn_test.get_answers(qid))
    cands = feat['top_candidates'][:10]
    scores = feat['node_scores']

    # Tim rank cua answer
    answer_ranks = []
    for i, c in enumerate(cands):
        if c in answers:
            answer_ranks.append((i+1, c, scores.get(c, 0)))

    print(f'\n[{qid}] {feat["question"]}')
    print(f'  Gold: {answers}')
    print(f'  Top-1: {cands[0]} (score={scores.get(cands[0],0):.4f})')
    if answer_ranks:
        for rank, ans, sc in answer_ranks:
            print(f'  Answer at rank {rank}: {ans} (score={sc:.4f})')
    else:
        print(f'  Answer NOT in top-10 candidates')

# === Thong ke nguyen nhan ===
print('\n' + '='*80)
print('=== ROOT CAUSE ANALYSIS ===')
print('='*80)

n_answer_not_in_graph = 0
n_answer_in_graph_low_rank = 0
n_multi_answer = 0
n_single_answer = 0

for qid in failures['f1_zero'] + failures['hit_zero_has_cand']:
    feat = gnn_test.get(qid)
    G = gnn_test.get_local_graph(qid)
    answers = gnn_test.get_answers(qid)
    answers_in_graph = [a for a in answers if a in G]

    if len(answers) > 1:
        n_multi_answer += 1
    else:
        n_single_answer += 1

    if not answers_in_graph:
        n_answer_not_in_graph += 1
    else:
        n_answer_in_graph_low_rank += 1

total_fail = len(failures['f1_zero']) + len(failures['hit_zero_has_cand'])
print(f'Total failures: {total_fail}')
print(f'  Answer NOT in subgraph: {n_answer_not_in_graph} ({100*n_answer_not_in_graph/total_fail:.1f}%)')
print(f'  Answer in graph, low rank: {n_answer_in_graph_low_rank} ({100*n_answer_in_graph_low_rank/total_fail:.1f}%)')
print(f'  Multi-answer questions: {n_multi_answer} ({100*n_multi_answer/total_fail:.1f}%)')
print(f'  Single-answer questions: {n_single_answer} ({100*n_single_answer/total_fail:.1f}%)')

# === Graph size distribution: fail vs success ===
fail_sizes = [gnn_test.get_local_graph(q).number_of_nodes() for q in failures['f1_zero'][:50]]
succ_sizes = [gnn_test.get_local_graph(q).number_of_nodes() for q in successes[:50]]
print(f'\nAvg graph size (F1=0 failures): {np.mean(fail_sizes):.0f} nodes')
print(f'Avg graph size (successes):     {np.mean(succ_sizes):.0f} nodes')


=== Failure Distribution ===
Success (F1>0 & Hit=1): 1241
F1=0 (total miss):      193
Hit=0 (wrong top-1):    204
No candidates:           1

=== TYPE 1: F1=0 (GNN khong tim duoc answer nao) ===

[WebQTest-1] what did james k polk do before he was president
  Seeds: ['James K. Polk'] (in graph: 1/1)
  Gold answers: ['United States Representative', 'Governor of Tennessee', 'Speaker of the United States House of Representatives']
  Top-5 GNN candidates: ['George M. Dallas', 'Politician', 'Lawyer', 'm.0g0vx']
  Answers in graph: 3/3
  >> REASON: Answers in graph but GNN ranked them too low

[WebQTest-3] who plays ken barlow in coronation street
  Seeds: ['Coronation Street'] (in graph: 1/1)
  Gold answers: ['William Roache']
  Top-5 GNN candidates: ['ITV Granada', 'ITV HD', 'Tony Warren', 'Drama']
  Answers in graph: 0/1
  >> REASON: Gold answers NOT in subgraph!

[WebQTest-12] who is governor of ohio 2011
  Seeds: ['Ohio'] (in graph: 1/1)
  Gold answers: ['John Kasich', 'Ted Strickland',